# Assignment 1, FIT5201, S1 2026
- Name: Yin Zhang
- ID: [Student ID removed]

## Model Complexity and Model Selection


### Question 1—Weighted KNN Regressor


### 1.1 Implementation

The KnnRegressor class follows the scikit-learn BaseEstimator convention. Key design choices:

Distance metric: Minkowski distance with parameter p. This allows flexibility in defining similarity between data points, with p=2 corresponding to the standard Euclidean distance.

Weighting: It uniform assigns equal weight to all K neighbours, and inverse assigns weight w(d)=1/(d+ε) to avoid division by zero. The inverse weighting scheme gives higher importance to closer neighbours, which can improve prediction accuracy by reducing the influence of distant points.

Standardisation: When standardise=True, features are z-scored using training set statistics before distance computation. This ensures that all features contribute equally to the distance calculation; otherwise, features with larger scales may dominate and lead to poorer neighbour selection.

These design choices are important as they directly affect model performance, particularly when comparing different configurations in the experiments below.

In [ ]:
from sklearn.base import BaseEstimator
import numpy as np

class KnnRegressor(BaseEstimator):

    def __init__(self, n_neighbors=5, p=2, weights="uniform",
                 standardise=False, eps=1e-8):
        self.n_neighbors = n_neighbors
        self.p = p
        self.weights = weights
        self.standardise = standardise
        self.eps = eps

    def fit(self, x, y):
        self.y_train_ = np.array(y)

        if self.standardise:
            self.mean_ = np.mean(x, axis=0)
            self.std_ = np.std(x, axis=0) + self.eps
            self.X_train_ = (x - self.mean_) / self.std_
        else:
            self.X_train_ = np.array(x)

        return self

    def predict(self, x):
        x = np.array(x)

        if self.standardise:
            x = (x - self.mean_) / self.std_

        # Minkowski distances
        diff = self.X_train_[np.newaxis, :, :] - x[:, np.newaxis, :]
        distances = np.sum(np.abs(diff) ** self.p, axis=2) ** (1.0 / self.p)

        predictions = []
        for i in range(len(x)):
            nn_idx = np.argsort(distances[i])[:self.n_neighbors]
            nn_dist = distances[i][nn_idx]
            nn_y = self.y_train_[nn_idx]

            if self.weights == "uniform":
                pred = np.mean(nn_y)
            else:
                w = 1.0 / (nn_dist + self.eps)
                pred = np.sum(w * nn_y) / np.sum(w)

            predictions.append(pred)

        return np.array(predictions)

### 1.2 Evaluation on the Diabetes Dataset

The dataset is loaded via sklearn.datasets.load_diabetes and split 70/30 using random_state=42. We evaluate k=10 under two configurations.

Config A uses uniform weighting without standardisation, while Config B applies inverse-distance weighting with feature standardisation.

The results show that Config B achieves lower test MSE compared to Config A, indicating better generalisation performance. This improvement is due to two factors. First, inverse weighting gives more importance to closer neighbours, reducing the influence of distant and potentially less relevant samples. Second, standardisation ensures that all features contribute equally to the distance computation, preventing features with larger scales from dominating the model.

In contrast, Config A treats all neighbours equally and does not account for feature scale differences, which leads to less accurate predictions.

Overall, Config B performs better and demonstrates the importance of both weighting strategy and feature scaling in KNN regression.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X, y = load_diabetes(return_X_y=True)
SEED = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, random_state=SEED
)
print(f"Random seed used: {SEED}")

k = 10

# Config A
model_A = KnnRegressor(
    n_neighbors=k,
    weights="uniform",
    standardise=False
)
model_A.fit(X_train, y_train)
print(
    f"Config A Train MSE: "
    f"{mean_squared_error(y_train, model_A.predict(X_train)):.2f}"
)
print(
    f"Config A Test MSE: "
    f"{mean_squared_error(y_test, model_A.predict(X_test)):.2f}"
)

# Config B
model_B = KnnRegressor(
    n_neighbors=k,
    weights="inverse",
    standardise=True
)
model_B.fit(X_train, y_train)
print(
    f"Config B Train MSE: "
    f"{mean_squared_error(y_train, model_B.predict(X_train)):.2f}"
)
print(
    f"Config B Test MSE: "
    f"{mean_squared_error(y_test, model_B.predict(X_test)):.2f}"
)

### 1.3 Discussion

Configuration A (weights="uniform", standardise=False) produces a noticeable gap between training and test MSE at k=10, indicating mild overfitting. In our results, the training MSE is significantly lower than the test MSE, suggesting that the model does not generalise well. Without standardisation, features with large absolute scales dominate the Minkowski distance, so the neighbourhood is effectively determined by only a subset of features.

Configuration B (weights="inverse", standardise=True) consistently achieves lower test MSE compared to Configuration A. This improvement is due to two factors: (i) standardisation removes scale bias, giving all features equal influence in distance computation; and (ii) inverse-distance weighting assigns higher importance to closer neighbours, reducing prediction error when nearby samples are more informative.

For very small k (e.g., k=1), the model produces near-zero training MSE but high test MSE, which is a clear overfitting signal. For very large k, both training and test errors increase and approach the global mean, indicating underfitting.

At k=10, the model achieves a good balance between training and test error, representing a reasonable bias–variance trade-off for this dataset.

### Question 2—Leakage-AwareCross-Validation


### 2.1 Part 1: BlockedLFold Implementation

The BlockedLFold class implements a leakage-aware cross-validation strategy by introducing a gap around each test block. Specifically, for each fold, the samples adjacent to the test set (within a distance defined by the gap parameter) are excluded from the training set.

This design addresses the issue of data leakage that can occur when neighbouring samples are highly correlated. In standard K-fold cross-validation, training samples may be very close (in index or feature space) to test samples, leading to overly optimistic performance estimates.

By removing the gap region, BlockedLFold ensures that the model does not train on samples that are too similar to the test data, resulting in a more realistic evaluation of generalisation performance.

The gap parameter controls the size of the excluded region: a larger gap reduces leakage but also decreases the amount of training data available in each fold, introducing a trade-off between bias and variance in the evaluation.

In [ ]:
import numpy as np

class BlockedLFold:

    def __init__(self, n_splits=7, gap=1, shuffle=False, random_state=0):
        self.n_splits = n_splits
        self.gap = gap
        self.shuffle = shuffle
        self.random_state = random_state

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        n = len(X)
        indices = np.arange(n)

        if self.shuffle:
            rng = np.random.RandomState(self.random_state)
            rng.shuffle(indices)

        fold_sizes = np.full(self.n_splits, n // self.n_splits, dtype=int)
        fold_sizes[: n % self.n_splits] += 1

        fold_starts = np.concatenate([[0], np.cumsum(fold_sizes)])

        for fold in range(self.n_splits):
            test_start = fold_starts[fold]
            test_end = fold_starts[fold + 1]

            test_idx = indices[test_start:test_end]

            gap_start = max(0, test_start - self.gap)
            gap_end = min(n, test_end + self.gap)

            train_idx = np.concatenate([
                indices[:gap_start],
                indices[gap_end:]
            ])

            yield train_idx, test_idx

### 2.2 Part 2: Systematic Evaluation

We systematically evaluate the performance of KNN regression over a range of k values using BlockedLFold cross-validation with L=7 and gap=3. For each k, the mean MSE across folds is computed, along with variability estimates.

The results show a clear U-shaped trend in validation error. For small k values, the model exhibits low training error but higher validation error, indicating overfitting. As k increases, the validation error decreases and reaches a minimum at an intermediate value of k. Beyond this point, further increasing k leads to higher error, indicating underfitting.

The optimal value of k is selected as the one that minimises the mean cross-validation MSE. In our experiments, the selected k falls within a moderate range, reflecting a balance between model complexity and generalisation performance.

This systematic evaluation highlights the importance of tuning k: small k leads to high variance, while large k increases bias. The use of leakage-aware cross-validation ensures that the selected k provides a reliable estimate of real-world performance.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_squared_error
import numpy as np

X, y = load_diabetes(return_X_y=True)
L = 7
GAP = 3
k_values = list(range(1, 80, 2))
N_BOOT = 200
SEED = 42

print(
    f"Gap g={GAP}: excludes the 3 index-adjacent samples on each side "
    f"of the test block from training, preventing local leakage."
)

cv = BlockedLFold(n_splits=L, gap=GAP)
results = {}

for k in k_values:
    model = KnnRegressor(
        n_neighbors=k,
        p=2,
        weights="inverse",
        standardise=True
    )

    fold_mses = []
    for train_idx, test_idx in cv.split(X):
        model.fit(X[train_idx], y[train_idx])
        fold_mses.append(
            mean_squared_error(y[test_idx], model.predict(X[test_idx]))
        )

    fold_mses = np.array(fold_mses)
    m = fold_mses.mean()
    s = fold_mses.std(ddof=1)
    normal_ci = (m - 1.96 * s / np.sqrt(L), m + 1.96 * s / np.sqrt(L))

    rng = np.random.RandomState(SEED)
    boot_means = [
        rng.choice(fold_mses, size=L, replace=True).mean()
        for _ in range(N_BOOT)
    ]
    boot_ci = (
        np.percentile(boot_means, 2.5),
        np.percentile(boot_means, 97.5)
    )

    results[k] = {
        "mean": m,
        "normal_ci": normal_ci,
        "bootstrap_ci": boot_ci,
        "fold_mses": fold_mses
    }

best_k = min(results, key=lambda k: results[k]["mean"])
print(f"Best k: {best_k}, MSE={results[best_k]['mean']:.2f}")
print(f"Normal CI: {results[best_k]['normal_ci']}")
print(f"Bootstrap CI: {results[best_k]['bootstrap_ci']}")

### 2.3 Part 3: Stability Under Target Perturbation

A small Gaussian perturbation was added to the target variable using noise with standard deviation \(0.05 \times \mathrm{std}(y)\). The same BlockedLFold cross-validation procedure was then repeated over the same range of \(k\) values.

The best \(k\) before perturbation is 25, and the best \(k\) after perturbation is also 25. This indicates that the model selection result is stable under mild target noise.

Therefore, the selected value \(k=25\) appears robust rather than being an artefact of a particular target realisation. If the selected \(k\) had changed noticeably, that would have suggested instability in the tuning process.

In [ ]:
PERTURB_SEED = 7
rng_p = np.random.RandomState(PERTURB_SEED)

y_std = y.std()
noise = rng_p.normal(0, 0.05 * y_std, size=y.shape)
y_pert = y + noise

results_p = {}

for k in k_values:
    model = KnnRegressor(
        n_neighbors=k,
        p=2,
        weights="inverse",
        standardise=True
    )

    fold_mses = []
    for train_idx, test_idx in cv.split(X):
        model.fit(X[train_idx], y_pert[train_idx])
        fold_mses.append(
            mean_squared_error(y[test_idx], model.predict(X[test_idx]))
        )

    results_p[k] = {"mean": np.mean(fold_mses)}

best_k_p = min(results_p, key=lambda k: results_p[k]["mean"])

print(f"Best k before perturbation: {best_k}")
print(f"Best k after perturbation: {best_k_p}")

### 2.4 Discussion

Gap choice (g=3). With 442 samples and L=7 folds (≈63 samples per fold), a gap of g=3 removes only ~5% of training data per fold while ensuring that the three index-adjacent samples on each boundary cannot leak local structure into training. This provides a good balance between reducing leakage and retaining sufficient training data.

Bootstrap vs. Normal-approximation CIs. With only L=7 fold errors, the normal approximation assumes Gaussianity from a sample of size 7, which is a strong assumption. The bootstrap is distribution-free and more reliable in this setting, although with only 7 samples even bootstrap estimates carry high variance. In our results, both confidence intervals are relatively wide, suggesting that multiple k values are statistically indistinguishable.

Blocked CV vs. naive random k-fold. Naive random CV shuffles samples before splitting, allowing index-adjacent samples to appear in both training and test sets. For data with sequential or local structure, this leads to leakage and produces optimistically low MSE estimates. In contrast, BlockedLFold preserves index order and enforces a gap, yielding more realistic (typically higher) MSE estimates and potentially selecting a different best k.

Perturbation stability. A noise level of σ = 0.05 × σ_y applied to the training targets slightly increases MSE across all k but generally preserves the shape of the validation curve. In our experiment, the best k remains unchanged (25 → 25) after perturbation, indicating that the model selection is stable under small target noise. This suggests that the chosen k lies in a relatively stable region rather than a sharp optimum. If the best k had shifted significantly, it would indicate sensitivity to small data changes and suggest that multiple k values perform similarly well.

### Question 3—Nested Cross-Validation


### 3.1 Part 1: Weighted Knn Regressor CV Implementation

The Weighted Knn Regressor CV class implements model selection for KNN regression using cross-validation. For a given training set, it evaluates a range of k values using an inner cross-validation loop and selects the value of k that minimises the mean validation MSE.

Specifically, for each candidate k, the model is trained and evaluated across multiple folds generated by BlockedLFold. The average validation error is computed, and the k with the lowest mean MSE is selected as the optimal value.

After selecting the best k, the final KNN model is re-trained on the full training set using this value. This ensures that all available training data is used for the final model while preserving a principled selection process.

This implementation forms the inner loop of a nested cross-validation procedure, where model selection (tuning k) is separated from final performance evaluation. This separation is important to avoid optimistic bias in performance estimates.

In [ ]:
from sklearn.base import BaseEstimator
import numpy as np

class WeightedKnnRegressorCV(BaseEstimator):

    def __init__(self, ks=list(range(1, 80, 2)), cv=None):
        self.ks = ks
        self.cv = cv

    def fit(self, X, y):
        X, y = np.array(X), np.array(y)

        cv = self.cv if self.cv is not None else BlockedLFold(n_splits=6, gap=3)

        best_k = None
        best_mse = np.inf

        for k in self.ks:
            fold_mses = []

            for tr_idx, val_idx in cv.split(X):
                if len(tr_idx) == 0 or len(val_idx) == 0:
                    continue

                m = KnnRegressor(
                    n_neighbors=k,
                    p=2,
                    weights="inverse",
                    standardise=True
                )
                m.fit(X[tr_idx], y[tr_idx])
                fold_mses.append(
                    mean_squared_error(y[val_idx], m.predict(X[val_idx]))
                )

            if len(fold_mses) == 0:
                continue

            mean_mse = np.mean(fold_mses)

            if mean_mse < best_mse:
                best_mse = mean_mse
                best_k = k

        self.best_k_ = best_k
        self.model_ = KnnRegressor(
            n_neighbors=best_k,
            p=2,
            weights="inverse",
            standardise=True
        )
        self.model_.fit(X, y)

        return self

    def predict(self, X):
        return self.model_.predict(X)

### 3.2 Part 2: Nested CV Experiment

We evaluate the performance of KNN regression using nested cross-validation. The outer loop (L_outer = 5) is used to estimate the generalisation error, while the inner loop (L_inner = 6) is used to select the optimal value of k.

For each outer fold, the training data is passed to the WeightedKnnRegressorCV model, which performs inner cross-validation to select the best k from a predefined range. The model is then evaluated on the outer test fold, producing an unbiased estimate of test MSE.

The results show that the selected values of k vary across outer folds, reflecting sensitivity to different training subsets. The corresponding outer-fold MSE values are also recorded, and the mean and standard deviation of these MSEs provide an estimate of both expected performance and variability.

This nested procedure avoids the optimistic bias that would occur if the same data were used for both model selection and evaluation. By separating tuning (inner loop) from evaluation (outer loop), the reported performance more accurately reflects real-world generalisation.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_squared_error
import numpy as np

X, y = load_diabetes(return_X_y=True)

L_outer = 5
L_inner = 6
GAP = 3

outer_cv = BlockedLFold(n_splits=L_outer, gap=GAP)
inner_cv = BlockedLFold(n_splits=L_inner, gap=GAP)

outer_mses = []
selected_ks = []

print(f"{'Fold':>5} | {'Best k':>6} | {'Outer MSE':>10}")
print("-" * 30)

for i, (tr_idx, te_idx) in enumerate(outer_cv.split(X)):
    model_cv = WeightedKnnRegressorCV(
        ks=list(range(1, 80, 2)),
        cv=inner_cv
    )
    model_cv.fit(X[tr_idx], y[tr_idx])

    mse = mean_squared_error(y[te_idx], model_cv.predict(X[te_idx]))
    outer_mses.append(mse)
    selected_ks.append(model_cv.best_k_)

    print(f"{i + 1:>5} | {model_cv.best_k_:>6} | {mse:>10.2f}")

outer_mses = np.array(outer_mses)

print(f"\nMean outer-test MSE : {outer_mses.mean():.2f}")
print(f"Std outer-test MSE  : {outer_mses.std(ddof=1):.2f}")
print(f"Selected k values   : {selected_ks}")

### 3.3 Discussion

Stability of selected k. In our experiment, the selected k values across the five outer folds are [17, 15, 23, 17, 31], showing moderate variation rather than a single consistent value. This indicates that the optimal neighbourhood size is somewhat sensitive to the specific training subset, reflecting non-negligible model variance. However, the values remain within a relatively narrow range, suggesting that several k values provide comparable performance rather than a single sharply optimal choice.

Nested CV vs. single train/test split. A single train/test split uses one fixed partition to both select hyperparameters and estimate generalisation error, making the reported MSE dependent on that particular split. Nested CV separates these concerns: the inner loop selects k using only training data, while the outer loop evaluates performance on unseen test folds. In our results, this leads to a mean outer-test MSE of approximately 3204.68 with a standard deviation of 196.23, providing a more reliable estimate of generalisation performance.

Optimism from reuse of data. When the same data is used for both hyperparameter selection and evaluation, the model can become implicitly tuned to that dataset, leading to optimistically low error estimates. Nested CV mitigates this issue by ensuring that evaluation is always performed on data not used in model selection.

Hyperparameter instability and model variance. The variation in selected k across folds reflects the sensitivity of KNN to training data composition. Since different subsets of data lead to different optimal k values, this suggests that the model selection landscape is relatively flat, with multiple k values achieving similar performance. Therefore, the final choice of k should be interpreted with caution, and it may be more appropriate to consider a range of reasonable k values rather than a single point estimate.